In [1]:
import os
import numpy as np
import pandas as pd
from typing import Dict, Any, Tuple

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# NEW: for plotting
import matplotlib.pyplot as plt

# Try to import joblib for saving scalers
try:
    import joblib
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False

# --------- TensorFlow / Keras ----------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# CONFIG
# =========================

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ASTRUSDT"]

MAIN_DIR = "crypto_data"           # where SYMBOL/SYMBOL_ML_ready.csv are saved
MODELS_DIR = "lstm_models"         # where trained models & scalers will be stored

TARGET_COL = "target_next_close"   # regression target (from your ML-ready CSVs)

TEST_SIZE = 0.2                    # 80% train, 20% test
SEQUENCE_LENGTH = 48               # 48 hours lookback window for LSTM
EPOCHS = 40
BATCH_SIZE = 64
RANDOM_STATE = 42                  # for reproducibility

# Candidate features: intersection with df.columns will be used
CANDIDATE_FEATURES = [
    # OHLCV
    "open", "high", "low", "close", "volume",

    # Technical indicators
    "SMA_20", "EMA_20", "RSI_14", "MACD", "MACD_signal",
    "BBM", "BBU", "BBL", "OBV",
    "return_1h", "return_3h", "return_6h",
    "vol_3h", "vol_6h", "vol_12h", "vol_24h",

    # Lag features
    "close_lag_1", "close_lag_3", "close_lag_6",
    "close_lag_12", "close_lag_24",
    "volume_lag_1", "volume_lag_3", "volume_lag_6",
    "volume_lag_12", "volume_lag_24",

    # Time features (we'll add if missing)
    "hour", "dayofweek", "day"
]

PRICE_COLS_FOR_OUTLIERS = ["open", "high", "low", "close", TARGET_COL]


# =========================
# UTILS
# =========================

def load_symbol_df(symbol: str) -> pd.DataFrame:
    """Load ML-ready CSV for a given symbol and sort by time."""
    symbol_dir = os.path.join(MAIN_DIR, symbol)
    csv_path = os.path.join(symbol_dir, f"{symbol}_ML_ready.csv")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found for {symbol}: {csv_path}")

    df = pd.read_csv(csv_path)

    # Ensure datetime and sort
    if "open_time" in df.columns:
        df["open_time"] = pd.to_datetime(df["open_time"])
        df = df.sort_values("open_time").reset_index(drop=True)

    return df


def clip_outliers_iqr(df: pd.DataFrame,
                      cols: list,
                      factor: float = 1.5) -> pd.DataFrame:
    """Winsorize selected numeric columns using IQR bounds."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        df[col] = df[col].clip(lower, upper)
    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add hour, dayofweek, day if 'open_time' is available."""
    df = df.copy()
    if "open_time" in df.columns and not pd.api.types.is_datetime64_any_dtype(df["open_time"]):
        df["open_time"] = pd.to_datetime(df["open_time"])

    if "open_time" in df.columns:
        df["hour"] = df["open_time"].dt.hour
        df["dayofweek"] = df["open_time"].dt.dayofweek
        df["day"] = df["open_time"].dt.day

    return df


def prepare_features_and_target(df: pd.DataFrame
                                ) -> Tuple[pd.DataFrame, pd.Series, list]:
    """
    - Adds time features
    - Clips outliers on price & target
    - Selects intersection of CANDIDATE_FEATURES and df.columns
    - Drops rows with NaNs in features/target
    """
    df = add_time_features(df)
    df = clip_outliers_iqr(df, PRICE_COLS_FOR_OUTLIERS)

    if TARGET_COL not in df.columns:
        raise ValueError(f"{TARGET_COL} not found in dataframe.")

    feature_cols = [c for c in CANDIDATE_FEATURES if c in df.columns]
    if not feature_cols:
        raise ValueError("No candidate features found in dataframe.")

    df_model = df[feature_cols + [TARGET_COL]].dropna().reset_index(drop=True)

    X = df_model[feature_cols]
    y = df_model[TARGET_COL]

    return X, y, feature_cols


def time_series_train_test_split(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = TEST_SIZE
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Deterministic time-based split (no shuffle)."""
    n = len(X)
    split_idx = int(n * (1 - test_size))

    X_train = X.iloc[:split_idx].copy()
    X_test = X.iloc[split_idx:].copy()
    y_train = y.iloc[:split_idx].copy()
    y_test = y.iloc[split_idx:].copy()

    return X_train, X_test, y_train, y_test


def build_lstm_sequences(
    X_scaled: np.ndarray,
    y_scaled: np.ndarray,
    seq_len: int
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Convert 2D [n_samples, n_features] into 3D [n_seq, seq_len, n_features]
    for LSTM. Each target is the value *after* the input sequence.
    """
    X_seq, y_seq = [], []
    for i in range(len(X_scaled) - seq_len):
        X_seq.append(X_scaled[i:i+seq_len])
        y_seq.append(y_scaled[i+seq_len])
    return np.array(X_seq), np.array(y_seq)


def evaluate_regression(y_true, y_pred) -> Dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MSE": mse, "MAE": mae, "R2": r2}


# =========================
# LSTM TRAINING
# =========================

def train_lstm_for_symbol(symbol: str) -> Dict[str, Any]:
    """
    Full LSTM pipeline for a single symbol:
    - Load data
    - Feature prep
    - Time split
    - Scale (MinMax) on train only
    - Build sequences
    - Train LSTM
    - Evaluate in original price scale
    - Save model, scalers, and prediction plot
    """
    print("\n=======================================")
    print(f"Training LSTM for symbol: {symbol}")
    print("=======================================")

    df = load_symbol_df(symbol)
    X, y, used_features = prepare_features_and_target(df)

    print(f"Total usable rows for {symbol}: {len(X)}")
    print("Using features:", used_features)

    # 1) Time-based split
    X_train_df, X_test_df, y_train_ser, y_test_ser = time_series_train_test_split(X, y)

    # 2) Scaling
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled_2d = scaler_X.fit_transform(X_train_df.values)
    X_test_scaled_2d = scaler_X.transform(X_test_df.values)

    y_train_scaled_2d = scaler_y.fit_transform(y_train_ser.values.reshape(-1, 1))
    y_test_scaled_2d = scaler_y.transform(y_test_ser.values.reshape(-1, 1))

    # 3) Sequences for train & test (no leakage)
    X_train_seq, y_train_seq = build_lstm_sequences(
        X_train_scaled_2d, y_train_scaled_2d, SEQUENCE_LENGTH
    )
    X_test_seq, y_test_seq = build_lstm_sequences(
        X_test_scaled_2d, y_test_scaled_2d, SEQUENCE_LENGTH
    )

    print(f"X_train_seq shape: {X_train_seq.shape}")
    print(f"X_test_seq shape : {X_test_seq.shape}")

    n_features = X_train_seq.shape[-1]

    # 4) Build model
    model = Sequential()
    model.add(Input(shape=(SEQUENCE_LENGTH, n_features)))
    model.add(LSTM(64, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32))
    model.add(Dropout(0.2))
    model.add(Dense(1))

    model.compile(loss="mse", optimizer="adam")

    es = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # 5) Train
    history = model.fit(
        X_train_seq, y_train_seq,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es],
        verbose=1
    )

    # 6) Predict on test sequences
    y_pred_seq_scaled = model.predict(X_test_seq, verbose=0)

    # 7) Inverse transform predictions and true values
    y_pred_seq = scaler_y.inverse_transform(y_pred_seq_scaled).ravel()
    y_test_seq_orig = scaler_y.inverse_transform(y_test_seq).ravel()

    metrics = evaluate_regression(y_test_seq_orig, y_pred_seq)

    print(f"\nLSTM Metrics for {symbol}:")
    print(f"  MSE: {metrics['MSE']:.4f}")
    print(f"  MAE: {metrics['MAE']:.4f}")
    print(f"  R2 : {metrics['R2']:.4f}")

    # 8) Prepare output directory
    os.makedirs(MODELS_DIR, exist_ok=True)
    symbol_model_dir = os.path.join(MODELS_DIR, symbol)
    os.makedirs(symbol_model_dir, exist_ok=True)

    # 9) Plot predictions vs actual and save
    plt.figure(figsize=(12, 5))
    plt.plot(y_test_seq_orig, label="Actual Price")
    plt.plot(y_pred_seq, label="Predicted Price")
    plt.title(f"LSTM Prediction vs Actual - {symbol}")
    plt.xlabel("Time Step")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(symbol_model_dir, f"{symbol}_lstm_predictions.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Saved prediction plot to: {plot_path}")

    # 10) Save model and scalers
    model_path = os.path.join(symbol_model_dir, f"{symbol}_lstm.h5")
    model.save(model_path)
    print(f"Saved LSTM model to: {model_path}")

    if HAS_JOBLIB:
        joblib.dump(scaler_X, os.path.join(symbol_model_dir, f"{symbol}_scaler_X.pkl"))
        joblib.dump(scaler_y, os.path.join(symbol_model_dir, f"{symbol}_scaler_y.pkl"))
        joblib.dump(used_features, os.path.join(symbol_model_dir, f"{symbol}_features.pkl"))
        print("Saved scalers and feature list.")
    else:
        print("joblib not available; scalers not saved.")

    return {
        "symbol": symbol,
        "metrics": metrics,
        "model_path": model_path,
        "plot_path": plot_path
    }


# =========================
# MAIN
# =========================

if __name__ == "__main__":
    all_results = {}

    for sym in SYMBOLS:
        try:
            result = train_lstm_for_symbol(sym)
            all_results[sym] = result["metrics"]
        except Exception as e:
            print(f"Error while processing {sym}: {e}")

    print("\n\n==================== LSTM SUMMARY ====================")
    for sym, m in all_results.items():
        print(f"\nSymbol: {sym}")
        print(f"  MSE: {m['MSE']:.4f}")
        print(f"  MAE: {m['MAE']:.4f}")
        print(f"  R2 : {m['R2']:.4f}")



Training LSTM for symbol: BTCUSDT
Total usable rows for BTCUSDT: 43792
Using features: ['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BBM', 'BBU', 'BBL', 'OBV', 'return_1h', 'return_3h', 'return_6h', 'vol_3h', 'vol_6h', 'vol_12h', 'vol_24h', 'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12', 'close_lag_24', 'volume_lag_1', 'volume_lag_3', 'volume_lag_6', 'volume_lag_12', 'volume_lag_24', 'hour', 'dayofweek', 'day']
X_train_seq shape: (34985, 48, 34)
X_test_seq shape : (8711, 48, 34)
Epoch 1/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - loss: 0.0041 - val_loss: 5.6702e-05
Epoch 2/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 49s 67ms/step - loss: 0.0017 - val_loss: 2.5175e-04
Epoch 3/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0013 - val_loss: 5.6332e-05
Epoch 4/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 24s 49ms/step - loss: 0.0010 - val_loss: 8.9455e-05
Epoch 5/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 38s 43ms/step - loss: 8.3831e-04 - val_loss

Saved prediction plot to: lstm_models\BTCUSDT\BTCUSDT_lstm_predictions.png
Saved LSTM model to: lstm_models\BTCUSDT\BTCUSDT_lstm.h5
Saved scalers and feature list.

Training LSTM for symbol: ETHUSDT
Total usable rows for ETHUSDT: 43792
Using features: ['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BBM', 'BBU', 'BBL', 'OBV', 'return_1h', 'return_3h', 'return_6h', 'vol_3h', 'vol_6h', 'vol_12h', 'vol_24h', 'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12', 'close_lag_24', 'volume_lag_1', 'volume_lag_3', 'volume_lag_6', 'volume_lag_12', 'volume_lag_24', 'hour', 'dayofweek', 'day']
X_train_seq shape: (34985, 48, 34)
X_test_seq shape : (8711, 48, 34)
Epoch 1/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - loss: 0.0046 - val_loss: 3.4929e-05
Epoch 2/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - loss: 0.0016 - val_loss: 1.3555e-04
Epoch 3/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - loss: 0.0013 - val_loss: 4.2374e-05
Epoch 4/40


Saved prediction plot to: lstm_models\ETHUSDT\ETHUSDT_lstm_predictions.png
Saved LSTM model to: lstm_models\ETHUSDT\ETHUSDT_lstm.h5
Saved scalers and feature list.

Training LSTM for symbol: BNBUSDT
Total usable rows for BNBUSDT: 43792
Using features: ['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BBM', 'BBU', 'BBL', 'OBV', 'return_1h', 'return_3h', 'return_6h', 'vol_3h', 'vol_6h', 'vol_12h', 'vol_24h', 'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12', 'close_lag_24', 'volume_lag_1', 'volume_lag_3', 'volume_lag_6', 'volume_lag_12', 'volume_lag_24', 'hour', 'dayofweek', 'day']
X_train_seq shape: (34985, 48, 34)
X_test_seq shape : (8711, 48, 34)
Epoch 1/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - loss: 0.0038 - val_loss: 2.3099e-04
Epoch 2/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - loss: 0.0017 - val_loss: 2.5033e-05
Epoch 3/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - loss: 0.0012 - val_loss: 5.1345e-05
Epoch 4/40


Saved prediction plot to: lstm_models\BNBUSDT\BNBUSDT_lstm_predictions.png
Saved LSTM model to: lstm_models\BNBUSDT\BNBUSDT_lstm.h5
Saved scalers and feature list.

Training LSTM for symbol: XRPUSDT
Total usable rows for XRPUSDT: 43792
Using features: ['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BBM', 'BBU', 'BBL', 'OBV', 'return_1h', 'return_3h', 'return_6h', 'vol_3h', 'vol_6h', 'vol_12h', 'vol_24h', 'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12', 'close_lag_24', 'volume_lag_1', 'volume_lag_3', 'volume_lag_6', 'volume_lag_12', 'volume_lag_24', 'hour', 'dayofweek', 'day']
X_train_seq shape: (34985, 48, 34)
X_test_seq shape : (8711, 48, 34)
Epoch 1/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - loss: 0.0073 - val_loss: 1.2597e-04
Epoch 2/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - loss: 0.0028 - val_loss: 1.0728e-04
Epoch 3/40
492/492 ━━━━━━━━━━━━━━━━━━━━ 20s 40ms/step - loss: 0.0022 - val_loss: 8.9691e-05
Epoch 4/40


Saved prediction plot to: lstm_models\XRPUSDT\XRPUSDT_lstm_predictions.png
Saved LSTM model to: lstm_models\XRPUSDT\XRPUSDT_lstm.h5
Saved scalers and feature list.

Training LSTM for symbol: ASTRUSDT
Total usable rows for ASTRUSDT: 24876
Using features: ['open', 'high', 'low', 'close', 'volume', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BBM', 'BBU', 'BBL', 'OBV', 'return_1h', 'return_3h', 'return_6h', 'vol_3h', 'vol_6h', 'vol_12h', 'vol_24h', 'close_lag_1', 'close_lag_3', 'close_lag_6', 'close_lag_12', 'close_lag_24', 'volume_lag_1', 'volume_lag_3', 'volume_lag_6', 'volume_lag_12', 'volume_lag_24', 'hour', 'dayofweek', 'day']
X_train_seq shape: (19852, 48, 34)
X_test_seq shape : (4928, 48, 34)
Epoch 1/40
280/280 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - loss: 0.0054 - val_loss: 0.0015
Epoch 2/40
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - loss: 0.0025 - val_loss: 7.8495e-04
Epoch 3/40
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - loss: 0.0021 - val_loss: 6.0997e-04
Epoch 4/40
28

Saved prediction plot to: lstm_models\ASTRUSDT\ASTRUSDT_lstm_predictions.png
Saved LSTM model to: lstm_models\ASTRUSDT\ASTRUSDT_lstm.h5
Saved scalers and feature list.


==================== LSTM SUMMARY ====================

Symbol: BTCUSDT
  MSE: 16325779.2784
  MAE: 1956.3239
  R2 : 0.9191

Symbol: ETHUSDT
  MSE: 4061.4914
  MAE: 43.3601
  R2 : 0.9842

Symbol: BNBUSDT
  MSE: 100.3989
  MAE: 7.3180
  R2 : 0.9917

Symbol: XRPUSDT
  MSE: 0.0002
  MAE: 0.0083
  R2 : 0.9947

Symbol: ASTRUSDT
  MSE: 0.0000
  MAE: 0.0009
  R2 : 0.9796
